# Activity 2 — One prompt, fifteen rows
### Prompt Engineering · Python PH × Hacktoberfest 2026

Write **one prompt**, run it on **15 support tickets**, and see where it breaks.

1. **Setup** — pick a provider, paste your key, run both cells.
2. **Prompt** — edit the prompt in step 2.
3. **Run** — try one ticket, then all 15.
4. **Score** — read the report, fix your prompt, re-run. Goal: **15/15**.

Free keys: [Gemini](https://aistudio.google.com/apikey) · [Groq](https://console.groq.com/keys) (switch between Groq and Gemini if rate limit hits)

Note: **Groq model is faster than the gemini model**

## 1 · Setup  *(edit the first cell only)*

In [ ]:
PROVIDER = "groq"  # @param ["gemini", "groq"]

GOOGLE_API_KEY = "ENTER YOUR GOOGLE API KEY HERE"  # @param {type:"string"}

GROQ_API_KEY = "ENTER YOUR GROQ API KEY HERE"  # @param {type:"string"} 

In [18]:
# Installs the SDKs, connects your key, loads the 15 tickets. No edits needed.
%pip -q install pandas google-genai groq

import os
import re
import statistics
import time
import pandas as pd

# --- Provider & model (set PROVIDER in the cell above) ---------------------
PROVIDER = globals().get("PROVIDER", "gemini").strip().lower()
if PROVIDER not in ("gemini", "groq"):
    raise ValueError(f'PROVIDER must be "gemini" or "groq", not "{PROVIDER}". Fix it in the cell above.')

MODELS = {"gemini": "gemini-3.8-flash", "groq": "openai/gpt-oss-20b"}   # small and cheap on purpose
MODEL = MODELS[PROVIDER]

# --- API key: cell above → environment variable → Colab Secrets → ask -------
KEY_NAME = {"gemini": "GOOGLE_API_KEY", "groq": "GROQ_API_KEY"}[PROVIDER]
KEY_URL = {"gemini": "https://aistudio.google.com/apikey", "groq": "https://console.groq.com/keys"}[PROVIDER]
API_KEY = (globals().get(KEY_NAME) or os.environ.get(KEY_NAME) or "").strip()
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = (userdata.get(KEY_NAME) or "").strip()
    except Exception:
        pass
if not API_KEY:
    import getpass
    API_KEY = getpass.getpass(f"Enter your {PROVIDER} API key (from {KEY_URL}): ").strip()
if not API_KEY:
    raise ValueError(f"No {PROVIDER} API key — put it in {KEY_NAME} in the cell above and re-run.")

if PROVIDER == "gemini":
    from google import genai
    gemini_client = genai.Client(api_key=API_KEY)
else:
    from groq import Groq
    groq_client = Groq(api_key=API_KEY)

def ask_model(prompt):
    """Send one prompt, return the text (or a short error note)."""
    try:
        if PROVIDER == "gemini":
            text = gemini_client.models.generate_content(model=MODEL, contents=prompt).text
        else:
            resp = groq_client.chat.completions.create(
                model=MODEL, messages=[{"role": "user", "content": prompt}])
            text = resp.choices[0].message.content
        return (text or "").strip()
    except Exception as e:
        msg = str(e)
        if "429" in msg or "quota" in msg.lower() or "rate limit" in msg.lower() or "RESOURCE_EXHAUSTED" in msg:
            if PROVIDER == "gemini":
                return "[error: rate limit — in the cell above, set PROVIDER to groq and re-run Setup]"
            return "[error: rate limit — wait a minute, then re-run]"
        return f"[error: {e}]"

# --- 15 support tickets (keep in sync with tickets.txt) ---------------------
TICKETS = [
    {"id": "TK-1041", "sender": "R. Bautista", "team": "Finance", "channel": "Email", "note": "Hi team, the monthly reconciliation file is throwing an error when I open it from the shared drive. Says the file is locked by another user but nobody on my team has it open. Happened twice this week. Can someone check who has the lock? Need this before Friday cutoff."},
    {"id": "TK-1042", "sender": "J. Mercado", "team": "Operations", "channel": "Viber", "note": "printer not working again"},
    {"id": "TK-1043", "sender": "A. Villanueva", "team": "HR", "channel": "Email", "note": "Requesting access to the new onboarding tracker for two new hires starting Monday. Names are in the attached sheet. Also please confirm if the old tracker will be archived or stay live in parallel."},
    {"id": "TK-1044", "sender": "M. Santos", "team": "Sales", "channel": "Helpdesk", "note": "Two things. First, my laptop has been restarting randomly since the update last week, usually mid-call which is not great. Second, and unrelated, the CRM export is dropping the region column entirely so our regional numbers are unusable right now. Not sure if same team handles both, apologies if not."},
    {"id": "TK-1045", "sender": "D. Reyes", "team": "Operations", "channel": "Helpdesk", "note": "VPN keeps disconnecting roughly every 20 minutes on the new build. Rolled back to the previous client and it's stable, so it looks like a build issue rather than my connection. Two other people on the floor are seeing the same thing."},
    {"id": "TK-1046", "sender": "K. Lim", "team": "Finance", "channel": "Email", "note": "Following up on my earlier ticket about the expense portal timeout. Never mind, it sorted itself out this morning, looks like it was the maintenance window. Closing this out, no action needed. Thanks anyway."},
    {"id": "TK-1048", "sender": "C. Aquino", "team": "Sales", "channel": "Viber", "note": "quick one - is the client portal supposed to log you out after 15 mins? because ours does and the client is asking if that's configurable on their end or ours. what should i tell them?"},
    {"id": "TK-1049", "sender": "L. Garcia", "team": "Operations", "channel": "Email", "note": "The weekly ops dashboard has been showing stale data since Tuesday. Timestamp in the corner says last refresh was 3 days ago. Everything else on the page loads fine, it's just not pulling new numbers."},
    {"id": "TK-1050", "sender": "B. Cruz", "team": "HR", "channel": "Helpdesk", "note": "Hi, so this is a bit of a long one, apologies in advance. We've been trying to get the leave balances to reconcile between the HR system and what payroll has, and they've been off by anywhere from half a day to three days depending on the person. It started sometime after the carryover process in January, though I can't pin down exactly when because nobody flagged it until people started querying their balances in March. We've checked the carryover rules and they look right. We've also checked whether it's only people who took leave in December and it isn't, some people who took nothing are also off. Payroll thinks it might be a rounding thing on our side but I'm not convinced because the differences aren't consistent. Anyway, we need someone who actually understands the sync between the two systems to have a look, because we're now manually adjusting about 40 people a month and that is not sustainable heading into midyear."},
    {"id": "TK-1051", "sender": "N. Ocampo", "team": "Finance", "channel": "Email", "note": "Purchase order approval emails are going to spam for three people on the team. Already checked their filters and nothing obvious. Others on the same team receive them fine."},
    {"id": "TK-1052", "sender": "T. Navarro", "team": "Finance", "channel": "Viber", "note": "URGENT!!! HIGH PRIORITY!!! My desktop wallpaper went back to the default blue one after the update. Please fix ASAP!!!"},
    {"id": "TK-1054", "sender": "G. Fernandez", "team": "Sales", "channel": "Email", "note": "The quote template is pulling last quarter's pricing for two product lines. Verified against the current price list and it's definitely outdated. We've caught it manually so far but it's gone out to at least one client already."},
    {"id": "TK-1057", "sender": "E. Bautista", "team": "IT", "channel": "Helpdesk", "note": "Scheduled backup job failed silently three nights running. No alert fired, we only noticed during the weekly check. Job log shows it starting but no completion entry."},
    {"id": "TK-1059", "sender": "I. Corpuz", "team": "Sales", "channel": "Viber", "note": "hi! do we have a limit on how many users we can add to a client workspace? client wants to add 40 more and i don't want to promise something we can't do. also is there extra cost? thanks!"},
    {"id": "TK-1060", "sender": "O. Salazar", "team": "HR", "channel": "Helpdesk", "note": "The org chart tool is showing two people under the wrong manager after the reorg. Both transferred teams in August. Everyone else updated correctly."},
]

# --- Scoring rules for the step 4 report (see FACILITATOR_traps.md) ---------
LABELS = ["Summary", "Priority", "Next action"]
PRIORITIES = ["High", "Medium", "Low"]
MAX_WORDS = 60            # whole answer
MAX_SUMMARY_WORDS = 25    # summary line

# 7 trap tickets, built to break a loose prompt
TRAPS = {
    "TK-1042": "sparse",       # ask for details, don't guess a fix
    "TK-1044": "two issues",   # summarise only the first
    "TK-1046": "resolved",     # "No action required."
    "TK-1048": "question",     # keep it a question, don't answer it
    "TK-1050": "rambling",     # summary stays under 15 words
    "TK-1052": "loud",         # shouting isn't High priority
    "TK-1059": "question",
}
ASKS_FOR_DETAILS = (r"\b(ask|request|clarify|find out|get|gather|collect|obtain)\b.*\b(detail|info|more|which|what|model|location|error)"
                    r"|\b(ask|contact|reach out to|follow up with|message|call)\b.*\b(user|sender|requester|reporter|Mercado)"
                    r"|\bplease (provide|share|send)\b")
NO_ACTION = r"\bno (further )?action|\bnone\b|\bn/?a\b|not (needed|required)"

def trap_problem(ticket_id, fields):
    """How the answer fell for this ticket's trap, or None."""
    kind = TRAPS.get(ticket_id)
    summary, priority, action = (fields.get(label, "") for label in LABELS)
    if kind == "sparse" and action and not re.search(ASKS_FOR_DETAILS, action, re.I):
        return "guessed a fix for a near-empty ticket (ask the sender for details instead)"
    if kind == "two issues" and re.search(r"laptop|restart", summary, re.I) and re.search(r"CRM|export|region", summary, re.I):
        return "packed two issues into one summary (summarise only the first)"
    if kind == "resolved" and action and not re.search(NO_ACTION, action, re.I):
        return "gave a next action when none is needed"
    if kind == "question" and summary and "?" not in summary:
        return "answered or reworded the ticket's question (keep it as a question)"
    if kind == "rambling" and len(summary.split()) > 15:
        return "let a long ticket make a long summary (keep it under 15 words)"
    if kind == "loud" and priority and priority.strip(" .").capitalize() != "Low":
        return "let the sender's shouting set the priority (judge by business impact)"
    return None

def check(ticket_id, text):
    """Problems with one answer. Empty list = pass."""
    if text.startswith("[error"):
        return ["got no answer (API error) — re-run step 4"]
    fields, extra = {}, []
    for line in text.splitlines():
        clean = line.replace("*", "").replace("#", "").strip(" -•\t")
        label = next((l for l in LABELS if clean.lower().startswith(l.lower() + ":")), None)
        if label:
            fields[label] = clean[len(label) + 1:].strip()
        elif clean:
            extra.append(clean)

    missing = [label for label in LABELS if label not in fields]
    if len(missing) == len(LABELS):
        issues = ["not in the 'Summary: / Priority: / Next action:' format"]
    else:
        issues = [f"missing a '{label}:' line" for label in missing]
        if extra:
            issues.append("out of format (has lines besides Summary / Priority / Next action)")
    if "Priority" in fields and fields["Priority"].strip(" .").capitalize() not in PRIORITIES:
        issues.append("priority isn't just High, Medium or Low")
    if len(text.split()) > MAX_WORDS:
        issues.append(f"too long (over {MAX_WORDS} words)")
    elif len(fields.get("Summary", "").split()) > MAX_SUMMARY_WORDS:
        issues.append("summary is longer than one short sentence")
    trap = trap_problem(ticket_id, fields)
    if trap:
        issues.append(trap)
    return issues

def show_report(rows):
    """Print the score, then each problem with the tickets that have it."""
    results = {r["id"]: check(r["id"], r["output"]) for r in rows}
    passed = sum(1 for issues in results.values() if not issues)
    print("=" * 60)
    print(f"REPORT: {passed}/{len(rows)} correct")
    print("=" * 60)
    if passed == len(rows):
        print("✓ All answers pass. Nice prompt!")
        return

    failures = {}
    for tid, issues in results.items():
        for issue in issues:
            failures.setdefault(issue, []).append(tid)
    print("Incorrect findings:")
    for issue, ids in sorted(failures.items(), key=lambda kv: -len(kv[1])):
        where = "all of them" if len(ids) == len(rows) else ", ".join(ids)
        print(f"  ✗ {len(ids)} ticket{'s' if len(ids) != 1 else ''}: {issue}  ({where})")

    words = {r["id"]: len(r["output"].split()) for r in rows}
    median = statistics.median(words.values())
    odd = [tid for tid, n in words.items() if n > median * 1.6 or n < median * 0.6]
    if odd:
        print(f"\nOdd ones out by length (typical answer ≈ {median:.0f} words): {', '.join(odd)}")
    print(f"\nFix your prompt in step 2, then re-run step 4. Goal: {len(rows)}/{len(rows)}.")

print(f"Loaded {len(TICKETS)} tickets  ·  {PROVIDER}: {MODEL}")
print("Ready — edit the prompt in step 2.")


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Loaded 15 tickets  ·  gemini: gemini-3.8-flash
Ready — edit the prompt in step 2.


In [19]:
# SAMPLE TICKET

TICKETS[0]

{'id': 'TK-1041',
 'sender': 'R. Bautista',
 'team': 'Finance',
 'channel': 'Email',
 'note': 'Hi team, the monthly reconciliation file is throwing an error when I open it from the shared drive. Says the file is locked by another user but nobody on my team has it open. Happened twice this week. Can someone check who has the lock? Need this before Friday cutoff.'}

## 2 · Your prompt  *(edit this cell)*

Placeholders: `{note}` (the ticket text), `{sender}`, `{team}`, `{channel}`. Start with this vague version, then improve it.

In [20]:
PROMPT = """You are a support assistant. Here is a ticket:

{note}

Give a summary, a priority, and a next action."""

## 3 · Try it on ONE ticket

In [21]:
t = TICKETS[0]
out = ask_model(PROMPT.format(**t))
print(out)

issues = check(t["id"], out)
print("\n" + "-" * 60)
print(f"{t['id']}: " + ("✓ pass" if not issues else "✗ " + "; ".join(issues)))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]

------------------------------------------------------------
TK-1041: ✗ got no answer (API error) — re-run step 4


## 4 · Run all fifteen and get your score

Scan the **output** column for the odd ones out, then read the **report** under the table.
*The inconsistency was always in your prompt — you couldn't see it when you ran it once.*

**Look for:** 
- much longer or shorter answers 
- missing or reshuffled fields 
- a question *answered* instead of summarised 
- an action where none is needed 
- a shouty ticket marked High

Fix the prompt in step 2 and re-run until you hit **15/15**.

In [22]:
rows = []
for i, t in enumerate(TICKETS, 1):
    out = ask_model(PROMPT.format(**t))
    rows.append({"id": t["id"], "team": t["team"], "output": out})
    print(f"{i:2d}/{len(TICKETS)}  {t['id']}  done")
    time.sleep(1.2)   # stay under the free-tier rate limit

# Table of all answers (line breaks kept, so differences in shape show)
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(rows)
display(df.style.set_properties(subset=["output"], **{"white-space": "pre-wrap", "text-align": "left"}))

show_report(rows)

 1/15  TK-1041  done
 2/15  TK-1042  done
 3/15  TK-1043  done
 4/15  TK-1044  done
 5/15  TK-1045  done
 6/15  TK-1046  done
 7/15  TK-1048  done
 8/15  TK-1049  done
 9/15  TK-1050  done
10/15  TK-1051  done
11/15  TK-1052  done
12/15  TK-1054  done
13/15  TK-1057  done
14/15  TK-1059  done
15/15  TK-1060  done


,id,team,output
0,TK-1041,Finance,"**Summary:** The user is unable to access a monthly reconciliation file on the shared drive due to a persistent ""file locked by another user"" error, impacting a time-sensitive task ahead of a Friday cutoff. **Priority:** **High** (Time-sensitive financial task with a pending deadline, recurring blocker affecting team productivity). **Next Action:** Check the file server management console (Shared Folders > Open Files) to identify the user or orphaned session holding the lock, terminate the active lock, and confirm access with the user."
1,TK-1042,Operations,"[error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]"
2,TK-1043,HR,"[error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]"
3,TK-1044,Sales,"[error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]"
4,TK-1045,Operations,"[error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]"
5,TK-1046,Finance,"[error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]"
6,TK-1048,Sales,"Here is the ticket classification and recommended next steps: * **Summary:** The user is asking to confirm if the 15-minute client portal session timeout is standard behavior and whether the timeout duration can be customized either by the client or internally. * **Priority:** Low (General inquiry / Configuration question) * **Next Action:** Check internal product documentation to confirm the default session timeout policy and determine if/where the timeout duration can be adjusted (client admin settings vs. internal tenant configuration), then provide the agent with the appropriate answer and instructions to pass along to the client."
7,TK-1049,Operations,"[error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]"
8,TK-1050,HR,"**Summary:** Leave balances between the HR system and Payroll have been inconsistent (differing by 0.5 to 3 days) since the January carryover. The customer has ruled out carryover rules and December leave activity as causes. The issue currently requires approximately 40 manual adjustments per month, creating an unsustainable operational burden heading into midyear. **Priority:** **High (P2)** — While not a complete system outage, it directly impacts payroll/leave accuracy for dozens of employees, involves core integration functionality, and creates significant ongoing manual overhead. **Next Action:** Escalate the ticket to the Integrations / Technical Escalations team to review the sync logs and mapping rules between the HR system and Payroll. Reply to the customer acknowledging the escalation and request 3–5 specific employee IDs (including expected vs. actual balances in both systems) to serve as test cases for the investigation."
9,TK-1051,Finance,"[error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]"


REPORT: 0/15 correct
Incorrect findings:
  ✗ 8 tickets: priority isn't just High, Medium or Low  (TK-1041, TK-1048, TK-1050, TK-1052, TK-1054, TK-1057, TK-1059, TK-1060)
  ✗ 7 tickets: out of format (has lines besides Summary / Priority / Next action)  (TK-1041, TK-1048, TK-1050, TK-1054, TK-1057, TK-1059, TK-1060)
  ✗ 7 tickets: too long (over 60 words)  (TK-1041, TK-1048, TK-1050, TK-1054, TK-1057, TK-1059, TK-1060)
  ✗ 7 tickets: got no answer (API error) — re-run step 4  (TK-1042, TK-1043, TK-1044, TK-1045, TK-1046, TK-1049, TK-1051)
  ✗ 1 ticket: answered or reworded the ticket's question (keep it as a question)  (TK-1048)
  ✗ 1 ticket: let the sender's shouting set the priority (judge by business impact)  (TK-1052)

Odd ones out by length (typical answer ≈ 59 words): TK-1042, TK-1043, TK-1044, TK-1045, TK-1046, TK-1048, TK-1049, TK-1050, TK-1051, TK-1057

Fix your prompt in step 2, then re-run step 4. Goal: 15/15.


## 5 · Stretch  *(optional)*

Have the model check its own answers. Select the cell below, uncomment it (`Ctrl+/`), and run it.

In [23]:
# if "rows" not in globals() or not rows:
#     raise RuntimeError("No results to check yet — run step 4 first.")

# def critique(output):
#     q = f"""Here is a support-ticket summary:

# {output}

# In one short line: does it have exactly three lines labelled "Summary:", "Priority:" (High, Medium or Low)
# and "Next action:", and is it under {MAX_WORDS} words? If yes, reply "OK". If not, say what's wrong."""
#     return ask_model(q)

# for r in [r for r in rows if not r["output"].startswith("[error")][:5]:
#     print(r["id"], "->", critique(r["output"]))
#     time.sleep(1.2)   # stay under the free-tier rate limit